### Importing Libraries

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import ast

### Reading files

#### Posts - DataFrames

In [2]:
df = pd.read_csv(r'C:\Users\ADMIN\Desktop\ITDSIU21099_HoangVanManh\Fashion-Marketing-Automation-Solutions\notebooks\final_df.csv')

In [3]:
df = df.drop(columns=['keywords_comments'])
df.head()

,post_id,postUser,timestamp,combined_tokens,likesCount,commentsCount,score_segtiment,keywords_posts
0,1,maryleest,2023-06-02 16:34:43+00:00,"['cannes', '2023', 'kilianparis', 'kiliancanne...",32070,142,0.403500,"['white', 'kiliancannes', 'dress', 'kilian', '..."
1,2,tinaabeysekara,2023-01-01 07:49:16+00:00,"['clock', 'struck', 'midnight', 'ring', '2022'...",6565,86,0.316667,"['ever', 'medium', '2022', 'thank', 'year', 'c..."
2,3,maryleest,2023-05-29 18:57:51+00:00,"['famous', 'stairs', 'photo', 'gustave_durin',...",28936,123,0.400000,"['famous', 'brown', 'jewelry', 'yessayan', 'ma..."
3,4,stephaniebroek,2023-03-07 19:30:42+00:00,"['visualized', 'moment', 'many', 'times', 'fir...",4764,215,0.168889,"['moment', 'chanelofficial', 'fashion', 'flowe..."
4,5,maryleest,2023-05-23 21:11:12+00:00,"['got', 'witness', 'historical', 'moment', 'ci...",13379,123,0.401000,"['witness', 'moment', 'cannesfilmfestival', 'g..."


### Comments - DataFrames

In [4]:
comments_df = pd.read_csv(r'C:\Users\ADMIN\Desktop\ITDSIU21099_HoangVanManh\Fashion-Marketing-Automation-Solutions\notebooks\new_comments.csv')
comments_df = comments_df.drop(columns=['comments','preferences','emoji_present','clean_comments','tokens'])
comments_df.rename(columns={"tfidf_keywords": "keywords_comments"}, inplace=True)
comments_df.head()

,post_id,commentUser,timestamp,polarity,sentiment,keywords_comments
0,1,wilsonjunior5055,2023-06-05 12:27:24+00:00,0.00,neutral,"['please', 'amazing', 'beautiful', 'clapping_h..."
1,1,mariavmh,2023-06-04 16:46:40+00:00,0.00,neutral,"['amazing', 'beautiful', 'clapping_hands', 'de..."
2,1,mariavmh,2023-06-04 16:46:34+00:00,0.96,positive,"['amazing', 'beautiful', 'clapping_hands', 'de..."
3,1,stylemesoftly_,2023-06-04 15:44:54+00:00,0.24,positive,"['red_heartred_heartred_heart', 'amazing', 'be..."
4,1,jillviv,2023-06-04 15:06:03+00:00,0.70,positive,['smiling_face_with_hearteyessmiling_face_with...


In [5]:
user_profile = comments_df.groupby("commentUser").agg({
    "post_id": "count", # Số lượng bình luận
    "polarity": "mean",  # Điểm cảm xúc trung bình (avg_sentiment_score)
    "keywords_comments": lambda x: " ".join(x),  # Gộp từ khóa từ bình luận
}).reset_index()

user_profile = user_profile.rename(columns={'post_id': 'comment_count'})

In [6]:
user_profile.head()

,commentUser,comment_count,polarity,keywords_comments
0,021_alaniss,1,0.80,"['fire', 'amazing', 'beautiful', 'clapping_han..."
1,100ycientas,1,0.24,"['amazing', 'beautiful', 'clapping_hands', 'de..."
2,1072.official,1,0.52,"['amazing', 'smiling_face_with_hearteyes', 'be..."
3,12maaria34,1,0.00,"['amazing', 'beautiful', 'clapping_hands', 'de..."
4,130wlifestyle,1,0.60,"['amazing', 'beautiful', 'clapping_hands', 'de..."


In [7]:
def build_interaction_sequence(df):
    # Sắp xếp theo timestamp
    df_sorted = df.sort_values('timestamp')
    # Lấy thông tin cần thiết cho mỗi event
    return df_sorted[['post_id', 'timestamp', 'polarity']].to_dict('records')

df_sequence = comments_df.groupby('commentUser').apply(build_interaction_sequence).reset_index().rename(
    columns={0: 'interaction_sequence'}
)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_11072\2493308275.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_sequence = comments_df.groupby('commentUser').apply(build_interaction_sequence).reset_index().rename(


In [8]:
df_sequence.head()

,commentUser,interaction_sequence
0,021_alaniss,"[{'post_id': 583, 'timestamp': '2023-06-02 21:..."
1,100ycientas,"[{'post_id': 319, 'timestamp': '2023-03-29 14:..."
2,1072.official,"[{'post_id': 65, 'timestamp': '2023-04-07 16:0..."
3,12maaria34,"[{'post_id': 79, 'timestamp': '2023-05-05 21:4..."
4,130wlifestyle,"[{'post_id': 272, 'timestamp': '2023-06-06 01:..."


In [9]:
df_user_profile = user_profile.merge(df_sequence, on='commentUser', how='left')
df_user_profile.head()

,commentUser,comment_count,polarity,keywords_comments,interaction_sequence
0,021_alaniss,1,0.80,"['fire', 'amazing', 'beautiful', 'clapping_han...","[{'post_id': 583, 'timestamp': '2023-06-02 21:..."
1,100ycientas,1,0.24,"['amazing', 'beautiful', 'clapping_hands', 'de...","[{'post_id': 319, 'timestamp': '2023-03-29 14:..."
2,1072.official,1,0.52,"['amazing', 'smiling_face_with_hearteyes', 'be...","[{'post_id': 65, 'timestamp': '2023-04-07 16:0..."
3,12maaria34,1,0.00,"['amazing', 'beautiful', 'clapping_hands', 'de...","[{'post_id': 79, 'timestamp': '2023-05-05 21:4..."
4,130wlifestyle,1,0.60,"['amazing', 'beautiful', 'clapping_hands', 'de...","[{'post_id': 272, 'timestamp': '2023-06-06 01:..."


In [10]:
# Fill null values in polarity with 0
df_user_profile['polarity'] = df_user_profile['polarity'].fillna(0)
df_user_profile.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4089 entries, 0 to 4088
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   commentUser           4089 non-null   object 
 1   comment_count         4089 non-null   int64  
 2   polarity              4089 non-null   float64
 3   keywords_comments     4089 non-null   object 
 4   interaction_sequence  4089 non-null   object 
dtypes: float64(1), int64(1), object(3)
memory usage: 159.9+ KB


In [11]:
# Tạo từ điển ánh xạ post_id -> keywords_posts (hoặc combined_tokens nếu bạn muốn)
post_keywords_dict = df.set_index("post_id")["keywords_posts"].to_dict()

In [12]:
print(df_user_profile["interaction_sequence"].dtype)  # Kiểu dữ liệu Pandas
# Kiểm tra 5 giá trị đầu tiên
for i in range(5):
    val = df_user_profile["interaction_sequence"].iloc[i]
    print("Row", i, ":", repr(val), "| type =", type(val))

object
Row 0 : [{'post_id': 583, 'timestamp': '2023-06-02 21:12:48+00:00', 'polarity': 0.8}] | type = <class 'list'>
Row 1 : [{'post_id': 319, 'timestamp': '2023-03-29 14:42:09+00:00', 'polarity': 0.24}] | type = <class 'list'>
Row 2 : [{'post_id': 65, 'timestamp': '2023-04-07 16:01:24+00:00', 'polarity': 0.52}] | type = <class 'list'>
Row 3 : [{'post_id': 79, 'timestamp': '2023-05-05 21:48:52+00:00', 'polarity': 0.0}] | type = <class 'list'>
Row 4 : [{'post_id': 272, 'timestamp': '2023-06-06 01:17:58+00:00', 'polarity': 0.6}] | type = <class 'list'>


In [13]:
df_user_profile["posts_sequence"] = df_user_profile["interaction_sequence"].apply(lambda x: [item["post_id"] for item in x])
# Bao nhiêu người có bình luận
print("Số người có bình luận:", df_user_profile.shape[0])

Số người có bình luận: 4089


In [14]:
#df_user_profile.to_csv('user_profile.csv', index=False)

In [15]:
# Check type of data
df_user_profile.dtypes

commentUser              object
comment_count             int64
polarity                float64
keywords_comments        object
interaction_sequence     object
posts_sequence           object
dtype: object

### User Profile - DataFrame

In [16]:
# Chuyển đổi chuỗi posts_sequences thành list
df_user_profile['posts_sequence'] = df_user_profile['posts_sequence'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
# Lọc ra những commentUser có posts_sequence > 3:
df_user_profile = df_user_profile[df_user_profile['posts_sequence'].apply(lambda x: len(x) > 3)]
df_user_profile.drop(columns=['interaction_sequence'], inplace=True)
df_user_profile.head(5)

,commentUser,comment_count,polarity,keywords_comments,posts_sequence
30,92thestudio,4,0.385,"['amazing', 'beautiful', 'clapping_hands', 'de...","[188, 216, 91, 57]"
191,alantrowe,10,0.229,"['amazing', 'beautiful', 'clapping_hands', 'de...","[822, 823, 815, 816, 812, 819, 810, 806, 757, ..."
209,alemarullo,4,0.490,"['amazing', 'beautiful', 'clapping_hands', 'de...","[84, 91, 129, 46]"
276,amberunraveled,4,0.350,"['amazing', 'beautiful', 'clapping_hands', 'de...","[255, 33, 32, 27]"
323,andrea_maria_tourino_19,5,0.224,"['amazing', 'beautiful', 'clapping_hands', 'de...","[168, 173, 170, 175, 251]"


In [17]:
print("Số người có bình luận trên 3 bài đăng:", df_user_profile.shape[0])

Số người có bình luận trên 3 bài đăng: 126


In [18]:
posts_df = df.copy()
posts_df.head()

,post_id,postUser,timestamp,combined_tokens,likesCount,commentsCount,score_segtiment,keywords_posts
0,1,maryleest,2023-06-02 16:34:43+00:00,"['cannes', '2023', 'kilianparis', 'kiliancanne...",32070,142,0.403500,"['white', 'kiliancannes', 'dress', 'kilian', '..."
1,2,tinaabeysekara,2023-01-01 07:49:16+00:00,"['clock', 'struck', 'midnight', 'ring', '2022'...",6565,86,0.316667,"['ever', 'medium', '2022', 'thank', 'year', 'c..."
2,3,maryleest,2023-05-29 18:57:51+00:00,"['famous', 'stairs', 'photo', 'gustave_durin',...",28936,123,0.400000,"['famous', 'brown', 'jewelry', 'yessayan', 'ma..."
3,4,stephaniebroek,2023-03-07 19:30:42+00:00,"['visualized', 'moment', 'many', 'times', 'fir...",4764,215,0.168889,"['moment', 'chanelofficial', 'fashion', 'flowe..."
4,5,maryleest,2023-05-23 21:11:12+00:00,"['got', 'witness', 'historical', 'moment', 'ci...",13379,123,0.401000,"['witness', 'moment', 'cannesfilmfestival', 'g..."


In [19]:
embedding_dim = 300
post_ids = posts_df["post_id"].unique()
post_embedding = {post_id: np.random.rand(embedding_dim) for post_id in post_ids}

In [20]:
from collections import defaultdict

# Tạo tập hợp tất cả các post_id xuất hiện trong dữ liệu người dùng
all_post_ids = set()
for seq in df_user_profile["posts_sequence"]:
    all_post_ids.update(seq)
all_post_ids = list(all_post_ids)
all_post_ids.sort()

# Tạo dictionary ánh xạ post_id sang index label
post2idx = {post_id: idx for idx, post_id in enumerate(all_post_ids)}
idx2post = {idx: post_id for post_id, idx in post2idx.items()}

print("Số lượng bài đăng trong tập label:", len(post2idx))

Số lượng bài đăng trong tập label: 444


In [21]:
# Xây dựng dataset dạng (input_sequence, target) 
# Lưu ý: Chúng ta chuyển input_sequence thành danh sách các vector embedding
input_sequences = []
target_labels = []

for seq in df_user_profile["posts_sequence"]:
    if len(seq) < 3:
        continue  # bỏ qua những chuỗi quá ngắn
    # Tạo các cặp từ chuỗi
    for i in range(1, len(seq)):
        input_seq = seq[:i]       # Lấy phần lịch sử
        target = seq[i]           # Bài đăng tiếp theo
        # Chuyển input_seq thành dãy vector embedding
        input_emb = [post_embedding[post_id] for post_id in input_seq if post_id in post_embedding]
        if len(input_emb) == 0:
            continue
        input_sequences.append(np.array(input_emb))
        target_labels.append(post2idx[target])

print("Số lượng cặp dữ liệu:", len(input_sequences))


Số lượng cặp dữ liệu: 689


In [22]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

class SequenceDataset(Dataset):
    def __init__(self, sequences, targets):
        self.sequences = sequences
        self.targets = targets
        
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        seq = torch.tensor(self.sequences[idx], dtype=torch.float)
        target = torch.tensor(self.targets[idx], dtype=torch.long)
        return seq, target

# Hàm collate để pad chuỗi về cùng độ dài
def collate_fn(batch):
    sequences, targets = zip(*batch)
    # sequences là danh sách tensor có kích thước [seq_len, embedding_dim]
    lengths = [s.size(0) for s in sequences]
    padded_sequences = pad_sequence(sequences, batch_first=True)  # Kích thước [batch, max_seq_len, embedding_dim]
    targets = torch.stack(targets)
    return padded_sequences, torch.tensor(lengths), targets

dataset = SequenceDataset(input_sequences, target_labels)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)

print("Số lượng batch:", len(dataloader))


Số lượng batch: 22


In [23]:
import torch.nn as nn

class BiLSTMRecommender(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, num_layers, num_classes, bidirectional=True):
        super(BiLSTMRecommender, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.bidirectional = bidirectional
        
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional
        )
        # Nếu bidirectional thì hidden_dim * 2
        lstm_out_dim = hidden_dim * 2 if bidirectional else hidden_dim
        self.fc = nn.Linear(lstm_out_dim, num_classes)
        
    def forward(self, x, lengths):
        # Chuyển lengths về CPU để pack_padded_sequence không báo lỗi
        lengths = lengths.cpu()
        packed_input = nn.utils.rnn.pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        packed_output, (hidden, cell) = self.lstm(packed_input)
        if self.bidirectional:
            hidden_final = torch.cat((hidden[-2], hidden[-1]), dim=1)
        else:
            hidden_final = hidden[-1]
        out = self.fc(hidden_final)
        return out

# Thiết lập các tham số
hidden_dim = 256
num_layers = 2
num_classes = len(post2idx)

model = BiLSTMRecommender(embedding_dim, hidden_dim, num_layers, num_classes, bidirectional=True)
print(model)


BiLSTMRecommender(
  (lstm): LSTM(300, 256, num_layers=2, batch_first=True, bidirectional=True)
  (fc): Linear(in_features=512, out_features=444, bias=True)
)


In [31]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
num_epochs = 50

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0
    all_preds = []
    all_targets = []

    for batch in dataloader:
        sequences, lengths, targets = batch
        sequences = sequences.to(device)
        lengths = lengths.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        outputs = model(sequences, lengths)  # outputs: [batch, num_classes]
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * sequences.size(0)

        # Metrics: collect predictions and targets
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_targets.extend(targets.cpu().numpy())

    avg_loss = epoch_loss / len(dataset)
    accuracy = accuracy_score(all_targets, all_preds)
    precision = precision_score(all_targets, all_preds, average='weighted', zero_division=0)
    recall = recall_score(all_targets, all_preds, average='weighted', zero_division=0)
    f1 = f1_score(all_targets, all_preds, average='weighted', zero_division=0)

    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}, "
          f"Acc: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")


Epoch 1/50, Loss: 6.1276, Acc: 0.0029, Precision: 0.0003, Recall: 0.0029, F1: 0.0005
Epoch 2/50, Loss: 6.0520, Acc: 0.0044, Precision: 0.0003, Recall: 0.0044, F1: 0.0005
Epoch 3/50, Loss: 5.9290, Acc: 0.0029, Precision: 0.0001, Recall: 0.0029, F1: 0.0002
Epoch 4/50, Loss: 5.7984, Acc: 0.0087, Precision: 0.0090, Recall: 0.0087, F1: 0.0048
Epoch 5/50, Loss: 5.5942, Acc: 0.0203, Precision: 0.0024, Recall: 0.0203, F1: 0.0040
Epoch 6/50, Loss: 5.4016, Acc: 0.0290, Precision: 0.0032, Recall: 0.0290, F1: 0.0053
Epoch 7/50, Loss: 5.2013, Acc: 0.0435, Precision: 0.0211, Recall: 0.0435, F1: 0.0204
Epoch 8/50, Loss: 4.9648, Acc: 0.0450, Precision: 0.0161, Recall: 0.0450, F1: 0.0178
Epoch 9/50, Loss: 4.6539, Acc: 0.0914, Precision: 0.0495, Recall: 0.0914, F1: 0.0510
Epoch 10/50, Loss: 4.2367, Acc: 0.1306, Precision: 0.0681, Recall: 0.1306, F1: 0.0771
Epoch 11/50, Loss: 3.6879, Acc: 0.1974, Precision: 0.1158, Recall: 0.1974, F1: 0.1290
Epoch 12/50, Loss: 3.1197, Acc: 0.2642, Precision: 0.1736, Reca

In [25]:
def predict_next_post(model, input_seq):
    model.eval()
    # input_seq: list các post_id
    # Chuyển đổi thành vector embedding
    seq_emb = [post_embedding[post_id] for post_id in input_seq if post_id in post_embedding]
    if len(seq_emb) == 0:
        return None
    seq_tensor = torch.tensor(seq_emb, dtype=torch.float).unsqueeze(0).to(device)  # [1, seq_len, embedding_dim]
    length_tensor = torch.tensor([len(seq_emb)])
    with torch.no_grad():
        output = model(seq_tensor, length_tensor)  # [1, num_classes]
        predicted_idx = torch.argmax(output, dim=1).item()
    predicted_post = idx2post[predicted_idx]
    return predicted_post

In [26]:
example_seq = [111]
predicted = predict_next_post(model, example_seq)
print("Bài đăng dự đoán:", predicted)

Bài đăng dự đoán: 251


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_11072\227806241.py:8: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\torch\csrc\utils\tensor_new.cpp:281.)
  seq_tensor = torch.tensor(seq_emb, dtype=torch.float).unsqueeze(0).to(device)  # [1, seq_len, embedding_dim]


In [27]:
def predict_next_posts(model, input_seq, num_predictions=5):
    model.eval()
    predicted_posts = []

    for _ in range(num_predictions):
        # Tạo embedding chuỗi hiện tại
        seq_emb = [post_embedding[post_id] for post_id in input_seq if post_id in post_embedding]
        if len(seq_emb) == 0:
            break
        seq_tensor = torch.tensor(seq_emb, dtype=torch.float).unsqueeze(0).to(device)
        length_tensor = torch.tensor([len(seq_emb)])

        with torch.no_grad():
            output = model(seq_tensor, length_tensor)
            predicted_idx = torch.argmax(output, dim=1).item()
        
        predicted_post = idx2post[predicted_idx]
        
        # Tránh lặp vô hạn nếu model cứ đoán ra 1 bài cũ
        if predicted_post in input_seq:
            break
        
        predicted_posts.append(predicted_post)
        input_seq.append(predicted_post)  # Thêm bài mới vào chuỗi để gợi ý tiếp

    return predicted_posts


In [28]:
example_seq = [111]
predicted = predict_next_posts(model, example_seq)
print("Bài đăng dự đoán:", predicted)

Bài đăng dự đoán: [251, 373, 389, 388, 386]
